In [ ]:
# ============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# ============================================================================
!pip install -q xgboost optuna shap scikit-learn pandas numpy matplotlib seaborn scipy openpyxl xlrd

import pandas as pd
import numpy as np
import warnings, json, pickle
from pathlib import Path
from datetime import datetime, timedelta
from typing import List # Add this import
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import xml.etree.ElementTree as ET

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# ── CONFIGURATION CHEMINS ────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
TERRAIN_ROOT = PROJECT_ROOT / 'src' / 'weather_ml_project' / 'data' / 'raw' / 'Données Météo'
ERA5_PATH = PROJECT_ROOT / 'data' / 'APIS' / 'interactiv' / 'InteractiveSheet_2026-05-06_15_23_36.xlsx'
OPENMETEO_PATH = PROJECT_ROOT / 'src' / 'weather_ml_project' / 'data' / 'APIS' / 'extraction_open_meteo_20260503_012319.csv'
NASA_PATH = PROJECT_ROOT / 'src' / 'weather_ml_project' / 'data' / 'APIS' / 'extraction_nasa_power_quotidienne_20260503_012720.csv'
PROCESSED = PROJECT_ROOT / 'data_processed'
PROCESSED.mkdir(exist_ok=True)

# Plage de prédiction finale (modifier selon besoin)
PRED_DATE_START = '2023-01-01'
PRED_DATE_END   = '2025-12-31'

print("✅ Configuration OK")
print(f"   Terrain : {TERRAIN_ROOT}")
print(f"   Output  : {PROCESSED}")
print(f"   Prédiction de {PRED_DATE_START} à {PRED_DATE_END}")

In [ ]:
import os

# List contents of TERRAIN_ROOT
terrain_root_contents = os.listdir(TERRAIN_ROOT)

if terrain_root_contents:
    print(f"Files and directories found in {TERRAIN_ROOT}:")
    for item in terrain_root_contents:
        print(f"- {item}")
else:
    print(f"The directory {TERRAIN_ROOT} is empty.")

In [ ]:
# ============================================================
# CELL 2 - PIPELINE TERRAIN ROBUSTE
# ============================================================

!pip install -q openpyxl xlrd lxml pandas -U

import pandas as pd
import numpy as np
from pathlib import Path
import xml.etree.ElementTree as ET
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CHARGEMENT ROBUSTE (EXCEL + XML)
# ============================================================
def load_excel_robust(file_path):
    try:
        with open(file_path, 'rb') as f:
            start = f.read(500)
    except:
        return None, None

    # XML
    if b'<?xml' in start:
        try:
            tree = ET.parse(file_path)
            root = tree.getroot()
            ns = {'ss': 'urn:schemas-microsoft-com:office:spreadsheet'}
            rows = []
            for row in root.findall('.//ss:Row', ns):
                row_data = []
                for cell in row.findall('ss:Cell', ns):
                    data = cell.find('ss:Data', ns)
                    row_data.append(data.text if data is not None else None)
                rows.append(row_data)
            if not rows:
                return None, None
            df = pd.DataFrame(rows)
            return df, 'xml'
        except Exception as e:
            print(f"      - XML error: {str(e)[:80]}")
            return None, None

    # Excel normal
    try:
        df = pd.read_excel(file_path, engine='openpyxl', header=None)
        return df, 'excel'
    except:
        pass
    try:
        df = pd.read_excel(file_path, engine='xlrd', header=None)
        return df, 'excel'
    except:
        pass
    return None, None

# ============================================================
# 2. NETTOYAGE PRÉLIMINAIRE (supprime lignes/colonnes vides)
# ============================================================
def clean_empty(df):
    df = df.dropna(how='all').dropna(axis=1, how='all').reset_index(drop=True)
    # Supprime les lignes qui ne contiennent que des None/NaN sur toutes les colonnes
    df = df.dropna(how='all')
    return df

# ============================================================
# 3. DÉTECTION EN-TÊTE + SÉPARATION DONNÉES
# ============================================================
def detect_header_and_data(df):
    """
    Parcourt les 15 premières lignes et prend la première ligne avec ≥ 3 non-vides
    comme en-tête. Retourne (header_row_values, data_df).
    """
    if len(df) == 0:
        return None, None
    for i in range(min(15, len(df))):
        row = df.iloc[i]
        if row.notna().sum() >= 3:   # seuil : au moins 3 colonnes non vides
            header = row.values
            data = df.iloc[i+1:].reset_index(drop=True)
            # Supprime les lignes entièrement vides qui pourraient rester
            data = data.dropna(how='all').reset_index(drop=True)
            return header, data
    return None, None

# ============================================================
# 4. NORMALISATION ET DÉDUPLICATION DES NOMS DE COLONNES
# ============================================================
def normalize_columns(df, header_values):
    cols = []
    for i, h in enumerate(header_values):
        h = str(h) if pd.notna(h) else f"col_{i}"
        # nettoyage
        h = h.lower()
        for a, b in [(' ','_'),('°',''),('(',''),(')',''),('/','_'),('-','_'),('[',''),(']',''),('\n',''),('\r',''), ('ç','c')]:
            h = h.replace(a, b)
        cols.append(h)

    # Rendre unique : col, col_1, col_2 ...
    seen = {}
    unique_cols = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            unique_cols.append(c)
        else:
            seen[c] += 1
            unique_cols.append(f"{c}_{seen[c]}")
    df.columns = unique_cols
    return df

# ============================================================
# 5. DÉTECTION COLONNE DATE
# ============================================================
def detect_date_column(df):
    for col in df.columns:
        sample = df[col].dropna().astype(str).head(20)
        if len(sample) == 0:
            continue
        parsed = pd.to_datetime(sample, errors='coerce', dayfirst=True)
        if parsed.notna().sum() >= max(3, len(sample)*0.5):
            return col
    return None

# ============================================================
# 6. PIPELINE PRINCIPAL
# ============================================================
def load_terrain_data(terrain_root, year_range=(2012,2025)):
    dfs = []
    ok, err, rows = 0, 0, 0

    print("="*100)
    print(f"📁 Dossier: {terrain_root}")
    print("="*100)

    # Lister les fichiers
    files = []
    for region in sorted(terrain_root.iterdir()):
        if not region.is_dir():
            continue
        for f in region.glob("*.xls"):
            try:
                if year_range[0] <= int(f.stem) <= year_range[1]:
                    files.append((region.name, f))
            except:
                pass

    print(f"\n📄 Total fichiers: {len(files)}\n")
    print("="*100)

    for i, (region, file) in enumerate(files, 1):
        print(f"[{i}/{len(files)}] {region:<30} | {file.name:<10}", end=" ")

        df_raw, engine = load_excel_robust(file)
        if df_raw is None or len(df_raw) == 0:
            print("❌ Vide")
            err += 1
            continue

        # Nettoyage des lignes/colonnes vides
        df_raw = clean_empty(df_raw)

        # Détection de l'en-tête et extraction des données
        header_vals, df_data = detect_header_and_data(df_raw)
        if header_vals is None or df_data is None or len(df_data) == 0:
            print("❌ Pas d'en-tête ou pas de données")
            err += 1
            continue

        # Appliquer les noms de colonnes normalisés et uniques
        df_data = normalize_columns(df_data, header_vals)

        # Recherche d'une colonne de date
        date_col = detect_date_column(df_data)
        if date_col is None:
            print("❌ Date non trouvée")
            err += 1
            continue

        # Conversion datetime
        df_data["datetime"] = pd.to_datetime(df_data[date_col], errors='coerce', dayfirst=True)
        df_data = df_data[df_data["datetime"].notna()]
        if len(df_data) == 0:
            print("❌ Aucune date valide")
            err += 1
            continue

        df_data["region_id"] = region
        df_data["source_file"] = file.name

        dfs.append(df_data)
        ok += 1
        rows += len(df_data)
        print(f"✅ {len(df_data):,} rows [{engine}]")

    print("\n"+"="*100)
    print(f"📊 OK: {ok} | ❌: {err} | ROWS: {rows:,}")
    print("="*100)

    if not dfs:
        return pd.DataFrame()

    # Concaténation sécurisée (ignore_index=True évite les conflits d'index)
    final = pd.concat(dfs, ignore_index=True)
    final = final.sort_values(["region_id", "datetime"]).reset_index(drop=True)

    print("✅ FINAL SHAPE:", final.shape)
    return final

# ============================================================
# 7. EXÉCUTION
# ============================================================
# Assurez-vous que TERRAIN_ROOT est défini avant d'appeler
# Exemple : TERRAIN_ROOT = Path("/content/drive/MyDrive/MyDrive/Données Météo")
df_terrain = load_terrain_data(TERRAIN_ROOT)

In [ ]:
print(TERRAIN_ROOT)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 4 : Chargement Données Satellite (ERA5, Open-Meteo, NASA POWER)
# ═══════════════════════════════════════════════════════════════════════════
import gc
from typing import Tuple, Optional

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Standardise les noms de colonnes pour faciliter la manipulation."""
    df.columns = [
        col.lower().replace(' ', '_').replace('.', '_').replace('-', '_')
        for col in df.columns
    ]
    df = df.rename(columns={
        't2m':                      'temperature',
        'total_precipitation_sum':  'precipitation',
        'tp':                       'precipitation',
        'date':                     'datetime',
    })
    return df


def convert_units(df: pd.DataFrame) -> pd.DataFrame:
    """Convertit les unités : Kelvin → Celsius pour les colonnes température."""
    temp_cols = [col for col in df.columns if 'temp' in col and df[col].max() > 100]
    for col in temp_cols:
        df[col] = df[col] - 273.15
    return df


def detect_geo_cols(df: pd.DataFrame, source_name: str = '') -> Tuple[Optional[str], Optional[str]]:
    """
    Détecte les colonnes lat/lon dans un DataFrame.
    Cherche des noms standard ('lat', 'latitude', 'lon', 'longitude')
    et des noms préfixés (ex: 'era5_lat', 'openmeteo_latitude') si source_name est fourni.
    """
    # Standard names
    possible_lat_names = ['lat', 'latitude']
    possible_lon_names = ['lon', 'longitude']

    # Add prefixed names based on common prefixes if source_name is available
    if source_name:
        possible_lat_names.extend([f'{source_name}_lat', f'{source_name}_latitude'])
        possible_lon_names.extend([f'{source_name}_lon', f'{source_name}_longitude'])
    # Also add known prefixed names from other sources to the search for generality
    possible_lat_names.extend(['nasa_lat', 'nasa_latitude', 'openmeteo_lat', 'openmeteo_latitude', 'era5_lat', 'era5_latitude'])
    possible_lon_names.extend(['nasa_lon', 'nasa_longitude', 'openmeteo_lon', 'openmeteo_longitude', 'era5_lon', 'era5_longitude'])

    # Remove duplicates and ensure order
    possible_lat_names = list(dict.fromkeys(possible_lat_names))
    possible_lon_names = list(dict.fromkeys(possible_lon_names))

    lat = next((c for c in df.columns if c in possible_lat_names), None)
    lon = next((c for c in df.columns if c in possible_lon_names), None)
    return lat, lon


def load_satellite_data(path: Path, source_name: str) -> pd.DataFrame:
    """Charge, standardise et préfixe les données satellite."""
    print(f"\n📡 Chargement {source_name}...")

    # Determine file type based on extension
    if path.suffix == '.csv':
        df = pd.read_csv(path)
    elif path.suffix == '.xlsx':
        df = pd.read_excel(path) # Use read_excel for .xlsx files
    else:
        raise ValueError(f"Unsupported file format for {source_name}: {path.suffix}")

    df = standardize_column_names(df)

    # NEW LOGIC: Identify and standardize geo columns BEFORE general prefixing
    actual_lat_col_in_df, actual_lon_col_in_df = detect_geo_cols(df, source_name='') # Generic search

    # Explicitly rename to 'latitude' and 'longitude' if found
    # and mark these as protected so they don't get 'source_name_' prefixed
    if actual_lat_col_in_df and actual_lon_col_in_df:
        if actual_lat_col_in_df != 'latitude': # Only rename if not already 'latitude'
            df = df.rename(columns={actual_lat_col_in_df: 'latitude'})
        if actual_lon_col_in_df != 'longitude': # Only rename if not already 'longitude'
            df = df.rename(columns={actual_lon_col_in_df: 'longitude'})
        print(f"  ✅ Geo-columns in {source_name} standardized to 'latitude', 'longitude'")
        final_protected_cols = {'datetime', 'latitude', 'longitude', 'source', 'region_id'}
    else:
        print(f"  ⚠️  No standard geo-columns found for {source_name}. Some geo-ops might fail if coordinates are missing.")
        # Fallback to original protected cols if no geo cols were found to standardize
        final_protected_cols = {'datetime', 'latitude', 'longitude', 'lat', 'lon', 'source', 'region_id'} # Fallback

    # ── Parse datetime ──────────────────────────────────────────────────────
    if 'datetime' not in df.columns:
        if all(c in df.columns for c in ['year', 'month', 'day']):
            df['datetime'] = pd.to_datetime(df[['year', 'month', 'day']])
        else:
            raise ValueError(f"[{source_name}] Impossible de construire la colonne 'datetime'. "
                             f"Colonnes disponibles : {list(df.columns)}")
    else:
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

    # Supprimer les lignes sans date valide
    n_before = len(df)
    df = df.dropna(subset=['datetime'])
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"  ⚠️  {n_dropped} lignes supprimées (datetime invalide)")

    # ── Conversions unités ──────────────────────────────────────────────────
    df = convert_units(df)

    # ── Marqueur source ─────────────────────────────────────────────────────
    df['source'] = source_name

    # ── Préfixage des colonnes (sauf colonnes protégées) ───────────────────
    rename_dict = {
        col: f"{source_name}_{col}"
        for col in df.columns
        if col not in final_protected_cols # Use the dynamically determined protected_cols
    }
    df = df.rename(columns=rename_dict)

    print(f"✅ {source_name} : {df.shape} | "
          f"Période : {df['datetime'].min()} → {df['datetime'].max()}")

    return df


# ── Chargement des 3 sources ────────────────────────────────────────────────
df_era5      = load_satellite_data(ERA5_PATH,      'era5')
df_openmeteo = load_satellite_data(OPENMETEO_PATH, 'openmeteo')
df_nasa      = load_satellite_data(NASA_PATH,      'nasa')

# Now, all dataframes should have 'latitude' and 'longitude' if they had geo data
# and these names will be consistent. So, we can set lat_col and lon_col directly.
lat_col = 'latitude'
lon_col = 'longitude'

# The previous block to detect lat_col, lon_col and raise ValueError is no longer needed
# as the above logic ensures consistent naming. If geo-cols were never found,
# assign_region_to_satellite will get 'latitude'/'longitude' but those columns won't exist in the DF,
# and it will raise an appropriate error. This is a more robust flow.

print(f"\n🌍 Colonnes géo utilisées : {lat_col}, {lon_col}")

# ── Sauvegarde ──────────────────────────────────────────────────────────────
df_era5.to_csv(PROCESSED / 'era5_processed.csv',         index=False)
df_openmeteo.to_csv(PROCESSED / 'openmeteo_processed.csv', index=False)
df_nasa.to_csv(PROCESSED / 'nasa_processed.csv',          index=False)

print("\n✅ Données satellite chargées et sauvegardées")
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 5 : Fusion Multi-Sources avec 3 Approches
#             Version corrigée + optimisée mémoire
# ═══════════════════════════════════════════════════════════════════════════
import gc
import numpy as np
from scipy.spatial.distance import cdist

# ── Configuration ───────────────────────────────────────────────────────────
METADATA_EXCLUSIONS = [
    'source_file', 'annee', 'col_0', 'col_6', 'col_7', 'col_8',
    'col_9', 'col_10', 'col_11', 'lat', 'lon', 'latitude', 'longitude'
]


# ═══════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def haversine_distances(sat_coords: np.ndarray,
                        terrain_coords: np.ndarray) -> np.ndarray:
    """
    Calcul distances haversine (km) entre points satellite et terrain.
    Plus précis que l'approximation euclidienne × 111.
    """
    R = 6371.0
    lat1 = np.radians(sat_coords[:, 0][:, None])
    lon1 = np.radians(sat_coords[:, 1][:, None])
    lat2 = np.radians(terrain_coords[:, 0][None, :])
    lon2 = np.radians(terrain_coords[:, 1][None, :])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def ensure_terrain_coords(df_terrain: pd.DataFrame,
                           lat_col: str, lon_col: str) -> pd.DataFrame:
    """
    FIX : Ajoute des coordonnées fictives au terrain si absentes.
    En production : remplacer par un vrai fichier de métadonnées stations.
    """
    if lat_col in df_terrain.columns and lon_col in df_terrain.columns:
        return df_terrain

    print("🔧 Coordonnées manquantes dans df_terrain — ajout de coords fictives (Maroc)...")
    unique_regions = df_terrain['region_id'].unique()
    n = len(unique_regions)

    rng = np.random.default_rng(42)
    dummy_lats = np.linspace(27.0, 36.0, n)
    dummy_lons = np.linspace(-13.0, -1.0, n)
    rng.shuffle(dummy_lats)
    rng.shuffle(dummy_lons)

    coords_df = pd.DataFrame({
        'region_id': unique_regions,
        lat_col:     dummy_lats,
        lon_col:     dummy_lons,
    })

    df_terrain = df_terrain.merge(coords_df, on='region_id', how='left')
    print(f"   ✅ {n} régions enrichies avec coordonnées fictives")
    return df_terrain


def assign_region_to_satellite(df_sat: pd.DataFrame,
                                df_terrain: pd.DataFrame,
                                lat_col: str, lon_col: str,
                                max_distance_km: float = 50) -> pd.DataFrame:
    """
    Assigne chaque point satellite à la région terrain la plus proche
    via distance haversine. Filtre les points > max_distance_km.
    """
    # FIX : vérifier que les colonnes géo existent dans df_sat
    for col in [lat_col, lon_col]:
        if col not in df_sat.columns:
            raise ValueError(
                f"Colonne '{col}' absente du DataFrame satellite. "
                f"Colonnes disponibles : {list(df_sat.columns)}"
            )

    # Coordonnées terrain moyennes par région
    terrain_coords = (
        df_terrain.groupby('region_id')[[lat_col, lon_col]]
        .mean()
        .reset_index()
    )

    # Coordonnées satellite uniques
    sat_unique = df_sat[[lat_col, lon_col]].drop_duplicates().copy()

    # FIX : distances haversine (au lieu d'euclidien × 111 imprécis)
    dist_matrix = haversine_distances(
        sat_unique[[lat_col, lon_col]].values,
        terrain_coords[[lat_col, lon_col]].values
    )

    closest_idx      = dist_matrix.argmin(axis=1)
    closest_dist_km  = dist_matrix.min(axis=1)

    sat_unique['region_id']   = terrain_coords.iloc[closest_idx]['region_id'].values
    sat_unique['distance_km'] = closest_dist_km

    # Filtrage distance maximale
    sat_unique = sat_unique[sat_unique['distance_km'] <= max_distance_km]

    # FIX : drop propre de region_id existant avant merge pour éviter doublons
    if 'region_id' in df_sat.columns:
        df_sat = df_sat.drop(columns=['region_id'])

    df_sat = df_sat.merge(
        sat_unique[[lat_col, lon_col, 'region_id']],
        on=[lat_col, lon_col],
        how='left'
    )

    n_assigned = df_sat['region_id'].notna().sum()
    n_total    = len(df_sat)
    print(f"   ✅ {n_assigned}/{n_total} points assignés "
          f"({100 * n_assigned / n_total:.1f}%)")

    return df_sat


def prepare_terrain(df_terrain: pd.DataFrame) -> pd.DataFrame:
    """Prépare le DataFrame terrain : sélection colonnes, typage datetime."""
    cols_to_keep = ['datetime', 'region_id'] + [
        c for c in df_terrain.columns
        if c not in ['datetime', 'region_id'] + METADATA_EXCLUSIONS
    ]
    # FIX : éviter les KeyError si certaines colonnes listées n'existent pas
    cols_to_keep = [c for c in cols_to_keep if c in df_terrain.columns]

    df_t = df_terrain[cols_to_keep].copy()
    df_t['datetime'] = pd.to_datetime(df_t['datetime'], errors='coerce')
    return df_t


def downcast_df(df: pd.DataFrame) -> pd.DataFrame:
    """Réduit les types numériques pour économiser la RAM."""
    for col in df.select_dtypes(include='float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df


# ═══════════════════════════════════════════════════════════════════════════
# ASSIGNATION RÉGIONS
# ═══════════════════════════════════════════════════════════════════════════

# FIX : s'assurer que df_terrain a des coordonnées avant l'assignation
df_terrain = ensure_terrain_coords(df_terrain, lat_col, lon_col)

# Add latitude and longitude to df_era5 by merging with df_terrain's coordinates
# This is necessary because ERA5 data in this context does not have inherent geo-coords
terrain_region_coords = df_terrain.groupby('region_id')[[lat_col, lon_col]].mean().reset_index()
if lat_col not in df_era5.columns:
    df_era5 = df_era5.merge(terrain_region_coords, on='region_id', how='left')
    print(f"  ✅ Coordinates from terrain assigned to df_era5 based on region_id.")

print("\n📍 Assignation des régions aux données satellite...")
print("  ERA5 :")
df_era5      = assign_region_to_satellite(df_era5,      df_terrain, lat_col, lon_col)
print("  Open-Meteo :")
df_openmeteo = assign_region_to_satellite(df_openmeteo, df_terrain, lat_col, lon_col)
print("  NASA :")
df_nasa      = assign_region_to_satellite(df_nasa,      df_terrain, lat_col, lon_col)


# ═══════════════════════════════════════════════════════════════════════════
# FUSION 3 APPROCHES
# ═══════════════════════════════════════════════════════════════════════════

def create_fusion_approaches(df_terrain: pd.DataFrame,
                              df_era5: pd.DataFrame,
                              df_openmeteo: pd.DataFrame,
                              df_nasa: pd.DataFrame) -> dict:
    """
    Crée 3 approches de fusion :
      A) Terrain + ERA5 uniquement
      B) Terrain + moyenne consensus (ERA5, Open-Meteo, NASA)
      C) Terrain + toutes sources séparément
    """
    MERGE_KEYS = ['datetime', 'region_id']

    # Typage datetime uniforme
    for df in [df_era5, df_openmeteo, df_nasa]:
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

    df_t = prepare_terrain(df_terrain)

    # ── Approche A : Terrain + ERA5 ─────────────────────────────────────────
    era5_cols = ['datetime', 'region_id'] + [
        c for c in df_era5.columns if c.startswith('era5_')
    ]
    # FIX : vérifier colonnes existantes
    era5_cols = [c for c in era5_cols if c in df_era5.columns]

    df_A = df_t.merge(
        df_era5[era5_cols],
        on=MERGE_KEYS, how='left'
    )
    df_A = downcast_df(df_A)
    print(f"\n✅ Approche A (Terrain + ERA5)       : {df_A.shape}")

    # ── Approche B : Terrain + Consensus ────────────────────────────────────
    # FIX : merge outer propre avec suffixes explicites
    df_sat = df_era5.merge(
        df_openmeteo, on=MERGE_KEYS, how='outer', suffixes=('', '_om')
    )
    df_sat = df_sat.merge(
        df_nasa, on=MERGE_KEYS, how='outer', suffixes=('', '_nasa')
    )

    # Calcul consensus pour variables communes
    common_vars = [
        'temperature', 'temp', 't2m',
        'precipitation', 'precip',
        'humidity', 'wind_speed', 'radiation'
    ]
    for var in common_vars:
        cols_var = [c for c in df_sat.columns if var in c.lower()]
        if len(cols_var) >= 2:
            df_sat[f'consensus_{var}'] = df_sat[cols_var].mean(axis=1, skipna=True)
            df_sat[f'std_{var}']       = df_sat[cols_var].std(axis=1,  skipna=True)

    df_B = df_t.merge(df_sat, on=MERGE_KEYS, how='left')
    df_B = downcast_df(df_B)
    print(f"✅ Approche B (Terrain + Consensus)  : {df_B.shape}")

    # Libération mémoire intermédiaire
    del df_sat
    gc.collect()

    # ── Approche C : Terrain + Toutes sources séparément ───────────────────
    df_C = df_t.merge(df_era5,      on=MERGE_KEYS, how='left')
    df_C = df_C.merge(df_openmeteo, on=MERGE_KEYS, how='left', suffixes=('', '_om'))
    df_C = df_C.merge(df_nasa,      on=MERGE_KEYS, how='left', suffixes=('', '_nasa'))
    df_C = downcast_df(df_C)
    print(f"✅ Approche C (Terrain + Toutes)     : {df_C.shape}")

    return {
        'A_terrain_era5':       df_A,
        'B_terrain_consensus':  df_B,
        'C_terrain_all':        df_C,
    }


# ── Exécution ───────────────────────────────────────────────────────────────
print("\n🔀 Création des 3 approches de fusion...")
fusion_datasets = create_fusion_approaches(
    df_terrain, df_era5, df_openmeteo, df_nasa
)

# ── Sauvegarde séquentielle avec libération mémoire ─────────────────────────
print("\n💾 Sauvegarde...")
for name, df in fusion_datasets.items():
    path = PROCESSED / f'fusion_{name}.csv'
    df.to_csv(path, index=False)
    print(f"   💾 {name} : {df.shape} → {path}")
    del df
    gc.collect()


gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 6 : Feature Engineering Complet + Lags Temporels  [RAM-safe]
# ═══════════════════════════════════════════════════════════════════════════

def create_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crée features temporelles cycliques — opère IN-PLACE pour éviter les copies
    """
    # Extraction base
    dt = df['datetime']
    df['year']       = dt.dt.year.astype('int16')
    df['month']      = dt.dt.month.astype('int8')
    df['day']        = dt.dt.day.astype('int8')
    df['hour']       = dt.dt.hour.astype('int8')
    df['dayofyear']  = dt.dt.dayofyear.astype('int16')
    df['weekofyear'] = dt.dt.isocalendar().week.astype('int8')
    df['dayofweek']  = dt.dt.dayofweek.astype('int8')

    # Encodage cyclique
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12).astype('float32')
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12).astype('float32')
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24).astype('float32')
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24).astype('float32')
    df['doy_sin']   = np.sin(2 * np.pi * df['dayofyear'] / 365).astype('float32')
    df['doy_cos']   = np.cos(2 * np.pi * df['dayofyear'] / 365).astype('float32')

    # Saisons → entiers int8 directement (évite get_dummies + float64)
    season_map = {12: 0, 1: 0, 2: 0,   # DJF
                   3: 1, 4: 1, 5: 1,   # MAM
                   6: 2, 7: 2, 8: 2,   # JJA
                   9: 3, 10: 3, 11: 3} # SON
    season_codes = df['month'].map(season_map).astype('int8')

    df['season_DJF'] = (season_codes == 0).astype('int8')
    df['season_MAM'] = (season_codes == 1).astype('int8')
    df['season_JJA'] = (season_codes == 2).astype('int8')
    df['season_SON'] = (season_codes == 3).astype('int8')

    # Interactions saisonnières
    df['hour_x_summer'] = (df['hour'] * df['season_JJA']).astype('float32')
    df['hour_x_winter'] = (df['hour'] * df['season_DJF']).astype('float32')

    print(f"   ✅ Features temporelles : {df.shape[1]} colonnes")
    return df


def create_meteorological_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crée features météorologiques avancées — in-place, pas de .copy()
    """
    temp_cols = [c for c in df.columns
                 if any(x in c.lower() for x in ['temp', 't2m', 'lst'])]
    rh_cols   = [c for c in df.columns
                 if any(x in c.lower() for x in ['humidity', 'rh', 'humidite'])]

    # Conversion numérique si nécessaire
    for col in temp_cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

    # VPD (uniquement si absent)
    if not any('vpd' in c.lower() for c in df.columns) and temp_cols and rh_cols:
        t  = df[temp_cols[0]].values
        rh = df[rh_cols[0]].values
        es = 0.611 * np.exp((17.27 * t) / (t + 237.3))
        df['vpd_calculated'] = (es - es * (rh / 100)).astype('float32')
        print(f"   ✅ VPD calculé depuis {temp_cols[0]} et {rh_cols[0]}")
        del es, t, rh  # libère mémoire intermédiaire

    # Amplitude thermique (si plusieurs sources temp)
    if len(temp_cols) >= 2:
        df['temp_amplitude']    = (df[temp_cols].max(axis=1)
                                   - df[temp_cols].min(axis=1)).astype('float32')
        df['temp_mean_sources'] = df[temp_cols].mean(axis=1).astype('float32')

    # Anomalie mensuelle
    if temp_cols and 'month' in df.columns:
        col = temp_cols[0]
        df[f'{col}_anomaly'] = (
            df[col] - df.groupby('month')[col].transform('mean')
        ).astype('float32')

    print(f"   ✅ Features météo : {df.shape[1]} colonnes")
    return df


def create_lag_features(df: pd.DataFrame, group_col: str = 'region_id') -> pd.DataFrame:
    """
    Lags temporels PAR RÉGION — in-place, types float32 dès la création
    """
    df.sort_values([group_col, 'datetime'], inplace=True)

    temp_cols = [c for c in df.columns
                 if any(x in c.lower() for x in ['temp', 't2m'])
                 and not any(k in c for k in ['anomaly', 'amplitude', 'lag', 'rolling'])]
    precip_cols = [c for c in df.columns
                   if any(x in c.lower() for x in ['precip', 'rain', 'pluie'])
                   and 'lag' not in c]

    for col in temp_cols[:2]:
        for lag in [1, 3, 6, 24]:
            df[f'{col}_lag_{lag}h'] = (
                df.groupby(group_col)[col].shift(lag).astype('float32')
            )

    for col in precip_cols[:1]:
        for lag in [1, 6, 24]:
            df[f'{col}_lag_{lag}h'] = (
                df.groupby(group_col)[col].shift(lag).astype('float32')
            )

    if temp_cols:
        df[f'{temp_cols[0]}_rolling_mean_24h'] = (
            df.groupby(group_col)[temp_cols[0]]
              .transform(lambda x: x.rolling(24, min_periods=1).mean())
              .astype('float32')
        )

    print(f"   ✅ Lags temporels : {df.shape[1]} colonnes")
    return df


def fill_missing_values_smart(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remplit les NaN — in-place, sans créer de colonnes intermédiaires
    """
    lag_cols     = [c for c in df.columns if 'lag' in c]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.difference(lag_cols)

    if lag_cols:
        df[lag_cols] = df[lag_cols].fillna(0)

    for col in numeric_cols:
        if df[col].isnull().any():
            df[col].fillna(df[col].mean(), inplace=True)

    print(f"   ✅ NaN remplis")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# Application — UN DATASET À LA FOIS pour éviter l'accumulation en RAM
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("🔧 FEATURE ENGINEERING  [mode RAM-safe : traitement séquentiel]")
print("=" * 80)

engineered_datasets = {}

for name, df in fusion_datasets.items():
    print(f"\n📊 {name}  ({df.shape[0]:,} lignes, {df.shape[1]} colonnes brutes)")

    # Toutes les étapes in-place sur le même objet
    df = create_temporal_features(df)
    df = create_meteorological_features(df)

    if 'region_id' in df.columns:
        df = create_lag_features(df, group_col='region_id')

    df = fill_missing_values_smart(df)

    # Float32 pour toutes les colonnes numériques (économie ~50 % vs float64)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].astype('float32')

    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"   ✅ {name} : {df.shape} | RAM : {mem_mb:.0f} MB")

    # Sauvegarde immédiate en parquet (4× plus léger que CSV)
    path = PROCESSED / f'engineered_{name}.parquet'
    df.to_parquet(path, index=False, engine='pyarrow', compression='snappy')
    print(f"   💾 Sauvegardé → {path.name}")

    engineered_datasets[name] = df

    # Libération explicite — important si RAM < 16 GB
    gc.collect()

print("\n✅ Feature engineering terminé pour les 3 approches")
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 7 OPTIMISÉE : Leakage Detection LIGHT (No RAM crash)
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd # Added this line
from typing import List, Tuple

def detect_and_remove_leakage_light(df: pd.DataFrame,
                                   target_col: str,
                                   correlation_threshold: float = 0.97,
                                   max_features: int = 40,
                                   sample_frac: float = 0.2) -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:
    """
    Version optimisée RAM :
    - Pas de matrice NxN
    - Corrélation feature vs target seulement
    - Sampling data
    - Limitation features
    """

    print(f"\n⚡ MODE LIGHT ACTIVÉ (anti-crash RAM)")

    # Sampling pour réduire RAM
    if len(df) > 50000:
        df = df.sample(frac=sample_frac, random_state=42)
        print(f"   🔻 Sampling appliqué : {len(df)} lignes")

    # Colonnes numériques
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if target_col not in numeric_cols:
        print(f"⚠️ Target non valide")
        return df, [], pd.DataFrame()

    # Limiter features
    numeric_cols = numeric_cols[:max_features]

    # Calcul corrélation simple (feature vs target uniquement)
    correlations = {}

    target_values = df[target_col]

    for col in numeric_cols:
        if col == target_col:
            continue
        try:
            correlations[col] = abs(df[col].corr(target_values))
        except:
            correlations[col] = 0

    correlations = pd.Series(correlations).sort_values(ascending=False)

    # Détection leakage
    leakage_features = correlations[correlations > correlation_threshold].index.tolist()

    # Rapport
    correlation_report = pd.DataFrame({
        'feature': correlations.index,
        'correlation_with_target': correlations.values,
        'is_leakage': correlations.index.isin(leakage_features)
    })

    # Suppression
    df_clean = df.drop(columns=leakage_features, errors='ignore')

    print(f"   ❌ Leakage détecté : {len(leakage_features)} features")
    print(f"   ✅ Shape : {df.shape} → {df_clean.shape}")

    return df_clean, leakage_features, correlation_report


def select_features_light(df: pd.DataFrame,
                         target_col: str,
                         max_features: int = 30) -> Tuple[pd.DataFrame, List[str]]:
    """
    Sélection ultra light pour éviter crash
    """

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    exclude = [target_col, 'datetime', 'region_id', 'year']
    feature_cols = [c for c in numeric_cols if c not in exclude]

    # Supprimer NaN full
    feature_cols = [c for c in feature_cols if not df[c].isnull().all()]

    # 🔻 LIMITATION FORTE
    feature_cols = feature_cols[:max_features]

    print(f"⚡ Features limitées à {len(feature_cols)}")

    return df, feature_cols


# ═══════════════════════════════════════════════════════════════════════════
# APPLICATION LIGHT
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("⚡ VERSION OPTIMISÉE ANTI-RAM")
print("=" * 80)

cleaned_datasets = {}
leakage_reports = {}

TARGETS = {
    'temperature': [c for c in engineered_datasets['A_terrain_era5'].columns
                    if 'temperature' in c.lower() or 't2m' in c.lower()][0],
    'precipitation': [c for c in engineered_datasets['A_terrain_era5'].columns
                      if 'precip' in c.lower() or 'rain' in c.lower()][0]
}

for name, df in engineered_datasets.items():

    print(f"\n📊 Dataset : {name}")

    df_clean = df.copy()
    all_leakage = []

    for target_name, target_col in TARGETS.items():

        if target_col not in df.columns:
            continue

        print(f"   🎯 Target : {target_name}")

        df_clean, leakage, report = detect_and_remove_leakage_light(
            df_clean,
            target_col
        )

        all_leakage.extend(leakage)
        leakage_reports[f"{name}_{target_name}"] = report

    cleaned_datasets[name] = df_clean

    print(f"   ✅ Final shape : {df.shape} → {df_clean.shape}")

gc.collect()